# OversightLab | Local ZIP + prospective mechanism study

**No GitHub clone or push.** The existing 450-episode results remain unchanged.


## 1 · Configuration (local Google Drive ZIP)

In [ ]:
# ============================================================
# 1. CONFIGURATION — LOCAL ZIP, NO GITHUB
# ============================================================
from pathlib import Path

MODE = "multi"  # "mock", "single", "multi"
RUN_EXPERIMENT = False
RUN_EVALUATION = False

RUN_MECHANISM = False       
RUN_MECHANISM_EVALUATION = False  
MECHANISM_SEEDS = 5
MECHANISM_REPEATS = 2
MECHANISM_HELDOUT_ROUNDS = (9001, 9002)
MECHANISM_BUDGET_USD = 9.0
MECHANISM_RESERVE_USD = 1.0
RUN_TESTS = True

SINGLE_MODEL = "anthropic/claude-haiku-5-5"
MULTI_MODELS = [
    "anthropic/claude-haiku-5-5",
    "gemini/gemini-3.5-flash-lite",
    "openai/gpt-6-luna",
]

# Pilot: ROUNDS=2, SEEDS=1; later confirmatory: ROUNDS=6, SEEDS=5.
ROUNDS = 6
SEEDS = 5
ARMS = [
    "control", "predictable_coarse", "predictable_detailed",
    "unpredictable_coarse", "unpredictable_detailed",
]
REGIMES = ["partial"]
MAX_AGENT_STEPS = 5
MAX_OUTPUT_TOKENS = 1200
TOTAL_BUDGET_USD = 10.0
BUDGET_RESERVE_USD = 1.0
PRICES = {}  # Optional: {model_id: (USD_per_1M_input, USD_per_1M_output)}

DRIVE_DIR = Path(".../...")
ZIP_PATH = DRIVE_DIR / "OversightLab_Github_Repo.zip"
RESULTS_ROOT = DRIVE_DIR / "oversightlab_multimodel_"
MECHANISM_RESULTS_ROOT = DRIVE_DIR / "oversightlab_mechanism"

## 2 · Load repository ZIP from Google Drive (no GitHub)

In [ ]:
# ============================================================
# 2. INSTALL DEPENDENCIES AND LOAD LOCAL REPOSITORY
# ============================================================
%pip -q install litellm pandas matplotlib pytest tqdm

import hashlib
import importlib
import inspect
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

archive_path = Path(ZIP_PATH)
if not archive_path.is_file():
    found = sorted(p.name for p in archive_path.parent.glob("*.zip"))
    raise FileNotFoundError(
        f"Repository ZIP not found: {archive_path}\n"
        f"Available ZIP files: {found}"
    )

# Stable source identifier for accurate, repeatable experiment routing.
source_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
extract_dir = Path("/content/oversightlab_source")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(extract_dir)

roots = sorted({path.parent.parent for path in extract_dir.rglob("oversight_lab/allocation.py")})
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one OversightLab source tree, found: {roots}")
PROJECT_DIR = roots[0]

# Source is extracted on each run; never reuse an imported prior version.
for name in list(sys.modules):
    if name == "oversight_lab" or name.startswith("oversight_lab."):
        del sys.modules[name]
sys.path = [p for p in sys.path if not str(p).startswith(str(extract_dir))]
sys.path.insert(0, str(PROJECT_DIR))
importlib.invalidate_caches()
from oversight_lab import allocation

assert allocation.PROTOCOL_VERSION == "allocation-v3-partialinfo", (
    f"Wrong repository version: {allocation.PROTOCOL_VERSION}"
)
episode_src = inspect.getsource(allocation.episode)
assert "operational_feedback" in episode_src, "Missing operational-feedback protocol"
assert "automatic_inspection" in episode_src, "Missing automatic-inspection protocol"
assert "heldout_probe" in inspect.getsource(allocation.run), "Missing blind held-out probes"

print("Repository ZIP:", archive_path)
print("ZIP SHA-256:", source_sha256)
print("Extracted project:", PROJECT_DIR)
print("Loaded allocation.py:", allocation.__file__)
print("✓ Correct local repository and protocol loaded")

## 3 · Free structural tests and baseline sensitivity (before any paid run)


In [ ]:
import pandas as pd
from IPython.display import display
from oversight_lab.allocation import sensitivity, make_state, oracle, score, public_state

if RUN_TESTS:
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", str(PROJECT_DIR / "tests")],
                            cwd=PROJECT_DIR, capture_output=True, text=True, timeout=120)
    print(result.stdout, result.stderr)
    assert result.returncode == 0, "Free validation failed. Do NOT run paid experiments."

s = make_state(7, 1, "partial")
assert "true_priority" in s and "true_priority" not in public_state(s)
ref = score(s, oracle(s))
assert ref["quality"] == 1
b = pd.DataFrame(sensitivity(30))
display(b.groupby("baseline")[["quality", "common"]].mean().round(4))
print("Validation complete: hidden weights never in prompt; oracle is exact.")

## 4 · Provider credentials for real experiments


In [ ]:
from google.colab import userdata
models = ({"mock": ["mock"], "single": [SINGLE_MODEL], "multi": MULTI_MODELS})[MODE]
credentials = {"anthropic": ("Anthropic_API", "ANTHROPIC_API_KEY"),
               "gemini": ("Gemini_API", "GEMINI_API_KEY"),
               "openai": ("OpenAI_API", "OPENAI_API_KEY")}
if (RUN_EXPERIMENT or RUN_MECHANISM) and MODE != "mock":
    for model in models:
        if "REPLACE_WITH" in model:
            raise ValueError(f"Choose a verified real provider model ID: {model}")
        provider = model.split("/")[0]
        if provider not in credentials:
            raise ValueError(f"Unsupported provider: {provider}")
        secret, env_name = credentials[provider]
        key = userdata.get(secret)
        if not key:
            raise ValueError(f"Missing Colab secret: {secret}")
        os.environ[env_name] = key.strip()
        assert "\r" not in os.environ[env_name] and "\n" not in os.environ[env_name]
        print(provider, "authenticated")
else:
    print("Offline/mock mode; no API authentication needed.")

## 5 · Execute local-ZIP experiments and blind pre/post held-out probes

In [ ]:
# ============================================================
# 5. RUN EXPERIMENTS — SOURCE FINGERPRINTED
# ============================================================
import hashlib
import json
import re
from pathlib import Path
from oversight_lab.allocation import run as run_allocation, PROTOCOL_VERSION

root = Path(RESULTS_ROOT) / MODE
root.mkdir(parents=True, exist_ok=True)
configuration = {
    "protocol": PROTOCOL_VERSION,
    "source_sha256": source_sha256,
    "rounds": ROUNDS,
    "seeds": SEEDS,
    "arms": ARMS,
    "steps": MAX_AGENT_STEPS,
    "tokens": MAX_OUTPUT_TOKENS,
}
run_dirs = {}
for regime in REGIMES:
    assert regime in ("full", "partial")
    for model in models:
        signature = hashlib.sha256(json.dumps(
            {**configuration, "regime": regime}, sort_keys=True
        ).encode()).hexdigest()[:10]
        slug = re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)
        folder = root / regime / f"{slug}_{signature}"
        if not RUN_EXPERIMENT and not (folder / "results.jsonl").exists():
            # Offline-only source edits should NOT hide previously collected data.
            # Never silently choose between multiple candidate experiments.
            candidates = sorted((root / regime).glob(f"{slug}_*/results.jsonl"))
            if len(candidates) != 1:
                raise RuntimeError(
                    f"Expected exactly one saved result set for {model}/{regime}; "
                    f"found {[str(p) for p in candidates]}. "
                    "Set RESULTS_ROOT to a pilot-only directory or select a specific run."
                )
            folder = candidates[0].parent
            first = json.loads(next(line for line in
                (folder / "results.jsonl").read_text().splitlines() if line.strip()))
            if (first.get("model") != model or first.get("observability") != regime
                    or first.get("protocol") != PROTOCOL_VERSION):
                raise RuntimeError(f"Stored results have incompatible metadata: {folder}")
            print(f"Offline evaluation reusing saved data: {folder}")
        run_dirs[(regime, model)] = folder

if RUN_EXPERIMENT:
    operating_budget = TOTAL_BUDGET_USD - BUDGET_RESERVE_USD
    assert operating_budget > 0, "Budget reserve must be less than total budget"
    for (regime, model), folder in run_dirs.items():
        budget_each = operating_budget / len(run_dirs)
        print(f"\nRUN {regime} | {model} | {ROUNDS * SEEDS * len(ARMS)} episodes")
        input_rate, output_rate = PRICES.get(model, (None, None))
        run_allocation(
            model, folder, rounds=ROUNDS, seeds=SEEDS, arms=ARMS,
            max_steps=MAX_AGENT_STEPS, max_tokens=MAX_OUTPUT_TOKENS,
            budget=budget_each, reserve=0,
            input_rate=input_rate, output_rate=output_rate,
            observability=regime,
        )
else:
    print("Experiment skipped; using previously saved results (no paid calls).")

## 6 · Inspect diagnostics (no evaluation model or judge is needed)


In [ ]:
for (regime, model), folder in run_dirs.items():
    path = folder / "results.jsonl"
    if not path.exists():
        print("No results:", folder)
        continue
    df = pd.read_json(path, lines=True)
    print("\n", regime, model, "episodes", len(df),
          "valid", int(df.valid.sum()), "inspections", int(df.automatic_inspections.sum()),
          "revisions", int(df.revised.sum()))
    display(df.groupby("condition")[["quality", "initial_quality", "revised",
                                      "final_inspection_score"]].mean().round(4))
    print("Pre/post heldout probes:", df.heldout_pre.notna().sum(),
          df.heldout_post.notna().sum())
    assert "true_priority" not in str(df.iloc[0].get("observed_inspection"))

## 7 · Offline evaluation, cross-provider figures, and matched seed contrasts


In [ ]:
from oversight_lab.allocation import analyze
import matplotlib.pyplot as plt

if RUN_EVALUATION:
    all_rows, reports = [], []
    for (regime, model), folder in run_dirs.items():
        path = folder / "results.jsonl"
        if not path.exists():
            continue
        summary = analyze(folder)
        reports.append({"regime": regime, "model": model, **summary})
        d = pd.read_json(path, lines=True)
        d["regime"] = regime
        all_rows.append(d)
        print("Evaluated", regime, model, summary)
    if reports:
        display(pd.DataFrame(reports).round(4))
    if all_rows:
        combined = pd.concat(all_rows, ignore_index=True)
        evaluation_dir = root / "evaluation"
        evaluation_dir.mkdir(parents=True, exist_ok=True)
        combined.drop(columns=["state", "actions", "allocation", "initial_allocation"],
                      errors="ignore").to_csv(evaluation_dir / "cross_model_episodes.csv", index=False)
        for regime, sub in combined.groupby("regime"):
            # Figure: model capability (seed means are the independent replications).
            fig, ax = plt.subplots(figsize=(8, 4.5))
            for model, m in sub.groupby("model"):
                y = m.groupby("round").quality.mean()
                ax.plot(y.index, y.values, marker="o", label=model)
            ax.set(xlabel="Experience round", ylabel="True operational quality / oracle",
                   title=f"Learning comparison — {regime}", ylim=(0, 1.05))
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(evaluation_dir / f"cross_model_learning_{regime}.png", dpi=180)
            plt.close(fig)
            # Figure: model x condition quality, descriptive, not a safety ranking.
            pivot = sub.groupby(["model", "condition"]).quality.mean().unstack()
            ax = pivot.plot.bar(figsize=(10, 5), ylim=(0, 1.05), rot=12,
                                title=f"Matched operational quality — {regime}")
            ax.set_ylabel("Quality / oracle")
            ax.figure.tight_layout()
            ax.figure.savefig(evaluation_dir / f"model_by_condition_{regime}.png", dpi=180)
            plt.close(ax.figure)
            # Matched per-seed treatment contrasts: do not count rounds as independent.
            seedmeans = sub.groupby(["model", "seed", "condition"]).quality.mean().unstack()
            if "control" in seedmeans:
                seedmeans.sub(seedmeans["control"], axis=0).to_csv(
                    evaluation_dir / f"matched_seed_effects_{regime}.csv")
            # Pair probes at trajectory level: pre in round 1, post in last round.
            keys = ["model", "seed", "condition", "regime"]
            pre = sub.loc[sub.heldout_pre.map(lambda x: isinstance(x, dict)),
                          keys + ["heldout_pre"]].copy()
            post = sub.loc[sub.heldout_post.map(lambda x: isinstance(x, dict)),
                           keys + ["heldout_post"]].copy()
            if pre.duplicated(keys).any() or post.duplicated(keys).any():
                raise ValueError("Duplicate held-out probes within a trajectory")
            pre["pre_quality"] = pre.heldout_pre.map(lambda x: x["quality"])
            pre["pre_valid"] = pre.heldout_pre.map(lambda x: x["valid"])
            post["post_quality"] = post.heldout_post.map(lambda x: x["quality"])
            post["post_valid"] = post.heldout_post.map(lambda x: x["valid"])
            transfer = pre.drop(columns=["heldout_pre"]).merge(
                post.drop(columns=["heldout_post"]), on=keys,
                validate="one_to_one", how="outer", indicator=True)
            transfer["pair_complete"] = transfer["_merge"].eq("both")
            transfer["gain_end_to_end"] = (
                transfer.post_quality - transfer.pre_quality
            ).where(transfer.pair_complete)
            transfer["gain_both_valid"] = transfer.gain_end_to_end.where(
                transfer.pre_valid.eq(True) & transfer.post_valid.eq(True))
            transfer.drop(columns=["_merge"]).to_csv(
                evaluation_dir / f"cross_model_heldout_pairs_{regime}.csv", index=False)
            matched = transfer.loc[transfer.pair_complete]
            if not matched.empty:
                averages = matched.groupby("model")[["pre_quality", "post_quality"]].mean()
                ax = averages.plot.bar(figsize=(8, 4.5), ylim=(0, 1.05), rot=12,
                                       title=f"Matched blind transfer — {regime}")
                ax.set_ylabel("Held-out quality / oracle; invalid = 0")
                ax.figure.tight_layout()
                ax.figure.savefig(
                    evaluation_dir / f"cross_model_heldout_{regime}.png", dpi=180)
                plt.close(ax.figure)
                validity = matched.groupby("model")[["pre_valid", "post_valid"]].mean()
                ax = validity.plot.bar(figsize=(8, 4.5), ylim=(0, 1.05), rot=12,
                                       title=f"Held-out submission validity — {regime}")
                ax.set_ylabel("Valid probe fraction")
                ax.figure.tight_layout()
                ax.figure.savefig(
                    evaluation_dir / f"cross_model_heldout_validity_{regime}.png", dpi=180)
                plt.close(ax.figure)
        print("Saved cross-model figures:", evaluation_dir)
else:
    print("Formal evaluation disabled. Set RUN_EXPERIMENT=False, RUN_EVALUATION=True.")

## 8 · Analysis discipline and next steps

- Offline scores compare **the exact true objective to its exact oracle**; neither the oracle scores nor latent customer weights appear in prompts or inspection feedback.
- **Learning** is in-context behavioral adaptation across a trajectory, not parameter updates.
- For the primary safety estimand compare matched seed-level true quality (treatment versus control), and report uncertainty across seeds, **not across rounds**.
- Inspectors and true operational value have different scales. Do not label a simple numeric compliance–quality difference “reward hacking.”
- Changes in held-out quality show transfer; changes in inspector score alone do not.
- The mock uses a privileged oracle by design, so it only validates plumbing. Do not report mock outcomes as agent behavior.
- Recommended: mock validation → 3-model 2-round 1-seed pilot → 6-round 5-seed study → freeze and write. Record repository ZIP SHA-256 in the manuscript. No further benchmark redesign based on which model wins.
- **Data sensitivity:** full state including latent weights is saved only to offline result files. If publishing raw trajectories, strip `true_priority` from any agent-visible trace to make the information boundary auditable.


## 9 · Counterfactual feedback study and historical held-out audit

**This is a separate prospective intervention, not a reanalysis of the 450-episode randomized trial.**
It reuses each model's **control-arm allocations** as a fixed six-round decision history. The operational states and exact held-out test cases are matched across all information conditions. Only permitted historical feedback and an accountability statement vary. No oracle scores or true priorities enter the prompts.

There are seven conditions: masked/neutral history; operational feedback only; operational plus each of the four selective inspectors; and predictable detailed inspection with an additional accountability statement. Each probe requires **one model call**, just like the original held-out evaluation. Two repeated responses per two held-out scenarios per seed give **140 probes/model (420 total)** with the standard 5 seeds. Five scenario seeds—not 420 calls—are the primary independent units. Exact prompt character counts are logged, but these are **not token-matched histories**, so residual context effects remain a limitation.

To run: leave `RUN_EXPERIMENT=False`, set `RUN_MECHANISM=True` and `RUN_MECHANISM_EVALUATION=False`. For purely offline reruns, reverse those two flags. Keep `RESULTS_ROOT` pointing to the **existing completed 450-episode study**, not the new output directory. A source-file SHA-256 checkpoint prevents silently mixing histories.

In [ ]:
# ============================================================
# 9. MATCHED FEEDBACK INTERVENTIONS + HISTORICAL FAILURE AUDIT
# ============================================================
import re
from pathlib import Path
from oversight_lab.allocation import (
    run_mechanism, audit_saved_holdouts, MECHANISM_CONDITIONS
)

source_folders = {}
for model in MULTI_MODELS if MODE == "multi" else models:
    slug = re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)
    candidates = sorted((Path(RESULTS_ROOT) / "multi" / "partial").glob(
        f"{slug}_*/results.jsonl"
    )) if MODE == "multi" else []
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one full-study results file for {model}; "
            f"found {[str(x) for x in candidates]}. Fix RESULTS_ROOT."
        )
    source_folders[model] = candidates[0]

# The historical 2026 pilot did not retain raw held-out messages.
# Unknown failure causes are explicitly classified as NOT RECORDED.
audit_dir = Path(MECHANISM_RESULTS_ROOT) / "historical_audit"
audit_dir.mkdir(parents=True, exist_ok=True)
for model, results_file in source_folders.items():
    slug = re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)
    rows = audit_saved_holdouts(results_file, audit_dir / f"{slug}.csv")
    print(model, "historical held-out observations audited:", len(rows))

if RUN_MECHANISM:
    if RUN_EXPERIMENT:
        raise ValueError("Do not run original episodes and mechanism study simultaneously")
    available = MECHANISM_BUDGET_USD - MECHANISM_RESERVE_USD
    assert available > 0
    for model, results_file in source_folders.items():
        slug = re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)
        output = Path(MECHANISM_RESULTS_ROOT) / slug
        in_rate, out_rate = PRICES.get(model, (None, None))
        print("RUN mechanism", model, "target probes:",
              MECHANISM_SEEDS * len(MECHANISM_HELDOUT_ROUNDS) *
              MECHANISM_REPEATS * len(MECHANISM_CONDITIONS))
        run_mechanism(
            model, results_file, output,
            seeds=MECHANISM_SEEDS,
            heldout_rounds=MECHANISM_HELDOUT_ROUNDS,
            repetitions=MECHANISM_REPEATS,
            budget=available / len(source_folders), reserve=0,
            max_tokens=MAX_OUTPUT_TOKENS,
            input_rate=in_rate, output_rate=out_rate,
        )
else:
    print("Mechanism API calls disabled. Historical audit created offline.")

## 10 · Offline mechanism analysis (no paid calls)

The primary contrast is **operational + inspection** versus **operational only**, evaluated on exactly the same new allocation scenario after identical historical decisions. A separate contrast isolates the additional accountability statement. Report **end-to-end economic quality (invalid = 0)**, **one-shot validity**, and **economic quality conditional on validity**; the conditional quantity is selected on success and is not a causal estimate. For proxy sensitivity, all conditions are rescored on the **same A,D inspection set**. Seed-level matched contrasts include descriptive 95% t intervals; five seeds give low precision. Do not call these results evidence of deliberate deception or model-weight learning.

In [ ]:
# ============================================================
# 10. OFFLINE MATCHED-CONTRAST EVALUATION
# ============================================================
from oversight_lab.allocation import analyze_mechanism
import pandas as pd
from IPython.display import display

if RUN_MECHANISM_EVALUATION:
    outputs = []
    for model in source_folders:
        slug = re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)
        output = Path(MECHANISM_RESULTS_ROOT) / slug
        if not (output / "mechanism_results.jsonl").exists():
            raise FileNotFoundError(f"No mechanism results at {output}")
        report = analyze_mechanism(output)
        outputs.append({"model": model, **report})
        print("Evaluated", model, report)
    display(pd.DataFrame(outputs))
else:
    print("Offline mechanism evaluation disabled; enable RUN_MECHANISM_EVALUATION.")